# 🚀 JalanCerdas AI — Retrain (All-in-One)
Run ALL cells. Training takes 30-50 min on GPU T4.

In [ ]:
!pip install -q ultralytics kagglehub pyyaml
import torch
print(f"✅ GPU: {torch.cuda.get_device_name(0)}" if torch.cuda.is_available() else "❌ NO GPU!")

## 1️⃣ Download & Prepare Dataset

In [ ]:
import kagglehub, os, shutil, yaml
import xml.etree.ElementTree as ET
from pathlib import Path

# === DOWNLOAD ===
print("📥 Downloading dataset...")
src_path = Path(kagglehub.dataset_download("habibiahmadim09/kerusakan-jalan"))
print(f"✅ Source: {src_path}")

# === CLONE REPO ===
if not Path("jalancerdas-ai").exists():
    !git clone -q https://github.com/Srjeff27/jalancerdas-ai.git

# === PREPARE ===
CLASS_NAMES = ["retak_memanjang","pengelupasan_lapisan_permukaan","lubang","retak_kulit_buaya","retak_blok","retak_pinggir"]
CID = {n:i for i,n in enumerate(CLASS_NAMES)}
dataset = Path("jalancerdas-ai/ai-model/dataset")
if dataset.exists(): shutil.rmtree(dataset)

for src_s, dst_s in [("train","train"),("valid","val"),("test","test")]:
    sd = src_path / src_s
    if not sd.exists():
        print(f"  ⚠️ {src_s}/ not found"); continue
    di = dataset/dst_s/"images"; dl = dataset/dst_s/"labels"
    di.mkdir(parents=True, exist_ok=True); dl.mkdir(parents=True, exist_ok=True)
    imgs = list(sd.rglob("*.jpg"))+list(sd.rglob("*.png"))+list(sd.rglob("*.jpeg"))
    xmls = {x.stem:x for x in sd.rglob("*.xml")}
    ni,nl = 0,0
    for img in imgs:
        shutil.copy2(img, di/img.name); ni+=1
        xml = xmls.get(img.stem)
        if xml:
            try:
                t=ET.parse(xml); r=t.getroot()
                w,h=int(r.find("size/width").text),int(r.find("size/height").text)
                ls=[]
                for o in r.findall("object"):
                    nm=o.find("name").text
                    if nm not in CID: continue
                    b=o.find("bndbox")
                    x1,y1=float(b.find("xmin").text),float(b.find("ymin").text)
                    x2,y2=float(b.find("xmax").text),float(b.find("ymax").text)
                    ls.append(f"{CID[nm]} {((x1+x2)/2)/w:.6f} {((y1+y2)/2)/h:.6f} {(x2-x1)/w:.6f} {(y2-y1)/h:.6f}")
                (dl/(img.stem+".txt")).write_text("\n".join(ls)); nl+=len(ls)
            except: pass
    print(f"  {dst_s}: {ni} images, {nl} labels")

# === FIX data.yaml ===
yp = Path("jalancerdas-ai/ai-model/configs/data.yaml")
dp = str(Path.cwd() / "jalancerdas-ai/ai-model/dataset")
with open(yp) as f: d=yaml.safe_load(f)
d['path'] = dp
with open(yp,'w') as f: yaml.dump(d,f,default_flow_style=False)

# === VERIFY ===
print(f"\n✅ data.yaml path: {dp}")
for s in ['train','val','test']:
    p=dataset/s/'images'
    n=len(list(p.glob('*'))) if p.exists() else 0
    print(f"  {s}/images: {n} files {'✅' if n>0 else '❌'}")

DATA_YAML = str(yp)
print(f"\n🚀 Ready to train! DATA_YAML={DATA_YAML}")

## 2️⃣ Train YOLOv8s (200 epochs)
⏱️ 30-50 min — **DON'T close tab!**

In [ ]:
from ultralytics import YOLO
model = YOLO("yolov8s.pt")
print("🚀 Training... 30-50 min\n")
results = model.train(data=DATA_YAML, epochs=200, batch=16, imgsz=640, device=0, patience=50, optimizer="SGD", lr0=0.01, lrf=0.01, mosaic=1.0, mixup=0.15, degrees=10.0, translate=0.1, scale=0.5, fliplr=0.5, copy_paste=0.1, close_mosaic=15, project="jalancerdas-ai/ai-model/runs", name="train_v2", exist_ok=True)
BEST = "jalancerdas-ai/ai-model/runs/train_v2/weights/best.pt"
print(f"\n✅ Done! {BEST}")

## 3️⃣ Evaluate

In [ ]:
model = YOLO(BEST)
r = model.val(data=DATA_YAML, device=0)
print(f"\n📊 mAP50={r.box.map50:.4f} | Prec={r.box.mp:.4f} | Recall={r.box.mr:.4f}")
for i,ap in enumerate(r.box.ap50): print(f"  {r.names[i]}: {ap:.4f}")

## 4️⃣ Export & Download

In [ ]:
model = YOLO(BEST)
ep = model.export(format="tflite", imgsz=640, half=True, simplify=True)
print(f"✅ TFLite: {ep} ({os.path.getsize(ep)/1024/1024:.1f} MB)")
from google.colab import files
files.download(ep)
files.download(BEST)
print("\n📥 Downloaded! Copy to mobile/assets/models/pothole_yolo.tflite")